# 📘 Tuần 4 – Mô hình 1D-CNN cho Phân loại Cử chỉ Tay Động
# (Phase 3: 1D Convolutional Neural Network for Temporal Gesture Recognition)

---

## 🎯 Mục tiêu

Ở tuần 2-3, chúng ta đã xây dựng **MLP Baseline** – mô hình chỉ nhìn vào **một frame duy nhất** để phân loại cử chỉ tĩnh.  
Tuần này, chúng ta nâng cấp lên **1D-CNN (1D Convolutional Neural Network)** – mô hình có khả năng **quan sát toàn bộ 15 frame** và trích xuất **đặc trưng chuyển động cục bộ (local temporal features)** bằng phép tích chập.

### Tại sao Conv1d lại phù hợp?

Mỗi mẫu cử chỉ là một chuỗi thời gian có shape `(15, 63)` – 15 bước thời gian, mỗi bước có 63 đặc trưng tọa độ.  
Khi ta **hoán vị (permute)** thành `(63, 15)`, Conv1d sẽ coi:
- **63 kênh đầu vào (channels):** mỗi kênh là một tọa độ landmark
- **15 bước thời gian (length):** chiều dài mà kernel trượt qua

Với `kernel_size=3`, bộ lọc tích chập sẽ trượt qua **3 frame liên tiếp**, học được các mẫu chuyển động ngắn hạn như:
- Gia tốc / giảm tốc của từng ngón tay
- Hướng di chuyển cục bộ (trái/phải/lên/xuống)
- Sự co/duỗi ngón tay trong 3 khung hình liền kề

Qua nhiều lớp Conv, mạng tổng hợp được **đặc trưng cấp cao hơn** – từ chuyển động cục bộ sang nhận diện toàn bộ quỹ đạo cử chỉ.

---
## 1. Import thư viện & Cấu hình

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import warnings
import time

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split

# ── Cấu hình hiển thị ──
warnings.filterwarnings('ignore')
matplotlib.rcParams.update({
    'figure.figsize': (12, 5),
    'font.size': 12,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'lines.linewidth': 2,
    'figure.dpi': 100
})

# ── Bảng màu chuyên nghiệp ──
COLORS = {
    'train_loss': '#2196F3',      # Blue
    'val_loss': '#F44336',        # Red
    'train_acc': '#4CAF50',       # Green
    'val_acc': '#FF9800',         # Orange
    'primary': '#1976D2',         # Dark Blue
    'secondary': '#E91E63',       # Pink
    'accent': '#00BCD4',          # Cyan
}

# ── Đường dẫn dự án ──
DATA_PATH = r'D:\WORKSAPCE\Monitor_AR\1_AI_Model_Training\Dataset\raw_data.csv'
MODEL_DIR = r'D:\WORKSAPCE\Monitor_AR\1_AI_Model_Training\Models'
BEST_MODEL_PATH = os.path.join(MODEL_DIR, 'best_cnn.pth')

os.makedirs(MODEL_DIR, exist_ok=True)

# ── Device ──
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'🖥️  Device: {device}')
if device.type == 'cuda':
    print(f'   GPU: {torch.cuda.get_device_name(0)}')
print(f'📂 Data:  {DATA_PATH}')
print(f'💾 Model: {BEST_MODEL_PATH}')

---
## 2. Tải và Khám phá Dữ liệu

In [ ]:
# ── Đọc CSV ──
df = pd.read_csv(DATA_PATH)
print(f'📊 Kích thước DataFrame: {df.shape}')
print(f'   Số cột: {df.shape[1]}')
print(f'   Số hàng: {df.shape[0]:,}')
print()

# ── Thống kê cơ bản ──
n_samples = df['sample_id'].nunique()
n_frames = df['frame_index'].nunique()
labels = df['label'].unique()
n_classes = len(labels)

print(f'🔢 Số mẫu (sample_id):    {n_samples}')
print(f'🎞️  Số frame / mẫu:         {n_frames}')
print(f'🏷️  Số lớp (classes):       {n_classes}')
print(f'   Nhãn: {sorted(labels)}')
print()

# ── Phân phối nhãn ──
label_counts = df.groupby('label')['sample_id'].nunique().sort_values(ascending=False)
print('📈 Phân phối nhãn (số mẫu):')
for lbl, cnt in label_counts.items():
    bar = '█' * (cnt * 40 // label_counts.max())
    print(f'   {lbl:>20s} │ {bar} {cnt}')

In [ ]:
# ── Xem trước vài dòng đầu ──
df.head(10)

---
## 3. Tiền xử lý Dữ liệu

### Pipeline:
1. **Xác định cột đặc trưng:** 63 cột landmark (x0, y0, z0, ..., x20, y20, z20)
2. **Reshape:** Gom các hàng cùng `sample_id` thành tensor `(15, 63)` 
3. **StandardScaler:** Chuẩn hóa toàn bộ đặc trưng về mean=0, std=1
4. **LabelEncoder:** Chuyển nhãn text sang số nguyên
5. **Train/Val/Test Split:** 70% / 15% / 15% với `stratify`

In [ ]:
# ── Bước 3.1: Xác định cột đặc trưng ──
meta_cols = ['sample_id', 'frame_index', 'label']
feature_cols = [c for c in df.columns if c not in meta_cols]
print(f'✅ Số cột đặc trưng: {len(feature_cols)}')
print(f'   Ví dụ: {feature_cols[:6]} ... {feature_cols[-3:]}')

In [ ]:
# ── Bước 3.2: Reshape thành mảng 3D ──
# Sắp xếp theo sample_id và frame_index để đảm bảo thứ tự
df_sorted = df.sort_values(['sample_id', 'frame_index']).reset_index(drop=True)

# Lấy nhãn cho mỗi sample (mỗi sample có 15 dòng cùng nhãn)
sample_ids = df_sorted['sample_id'].unique()
labels_per_sample = df_sorted.groupby('sample_id')['label'].first().loc[sample_ids].values

# Gom feature thành mảng (n_samples, 15, 63)
features_2d = df_sorted[feature_cols].values  # (n_samples * 15, 63)
X_all = features_2d.reshape(len(sample_ids), n_frames, len(feature_cols))  # (N, 15, 63)

print(f'✅ X_all shape: {X_all.shape}')       # (N, 15, 63)
print(f'   labels shape: {labels_per_sample.shape}')  # (N,)

In [ ]:
# ── Bước 3.3: StandardScaler ──
# Chuẩn hóa trên toàn bộ features (flatten thành 2D rồi reshape lại)
scaler = StandardScaler()
X_flat = X_all.reshape(-1, len(feature_cols))  # (N*15, 63)
X_flat_scaled = scaler.fit_transform(X_flat)
X_scaled = X_flat_scaled.reshape(X_all.shape)    # (N, 15, 63)

print(f'✅ Sau chuẩn hóa:')
print(f'   Mean ≈ {X_scaled.mean():.6f}')
print(f'   Std  ≈ {X_scaled.std():.6f}')

In [ ]:
# ── Bước 3.4: LabelEncoder ──
le = LabelEncoder()
y_all = le.fit_transform(labels_per_sample)
num_classes = len(le.classes_)

print(f'✅ Label Encoding:')
for i, cls_name in enumerate(le.classes_):
    count = (y_all == i).sum()
    print(f'   {i} → {cls_name} ({count} mẫu)')

In [ ]:
# ── Bước 3.5: Train / Val / Test Split (70/15/15) ──
X_train, X_temp, y_train, y_temp = train_test_split(
    X_scaled, y_all, test_size=0.30, random_state=42, stratify=y_all
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print(f'📦 Phân chia dữ liệu:')
print(f'   Train: {X_train.shape[0]:>5d} mẫu ({X_train.shape[0]/len(y_all)*100:.1f}%)')
print(f'   Val:   {X_val.shape[0]:>5d} mẫu ({X_val.shape[0]/len(y_all)*100:.1f}%)')
print(f'   Test:  {X_test.shape[0]:>5d} mẫu ({X_test.shape[0]/len(y_all)*100:.1f}%)')
print(f'   Tổng:  {len(y_all):>5d} mẫu')

---
## 4. Custom Dataset & DataLoader

In [ ]:
class GestureSequenceDataset(Dataset):
    """Dataset trả về chuỗi đầy đủ (15, 63) cho mỗi mẫu cử chỉ."""

    def __init__(self, X: np.ndarray, y: np.ndarray):
        """
        Args:
            X: mảng numpy shape (N, 15, 63)
            y: mảng numpy shape (N,) chứa nhãn đã encode
        """
        self.X = torch.FloatTensor(X)
        self.y = torch.LongTensor(y)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]  # (15, 63), scalar


# ── Tạo Dataset & DataLoader ──
BATCH_SIZE = 32

train_ds = GestureSequenceDataset(X_train, y_train)
val_ds   = GestureSequenceDataset(X_val,   y_val)
test_ds  = GestureSequenceDataset(X_test,  y_test)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  drop_last=False)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, drop_last=False)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, drop_last=False)

# ── Kiểm tra một batch ──
sample_X, sample_y = next(iter(train_loader))
print(f'✅ Batch kiểm tra:')
print(f'   X shape: {sample_X.shape}')   # (32, 15, 63)
print(f'   y shape: {sample_y.shape}')   # (32,)
print(f'   y values: {sample_y[:8].tolist()}')

---
## 5. Kiến trúc 1D-CNN

### Luồng dữ liệu trong mô hình:

```
Input: (Batch, 15, 63)
    │
    ▼ permute(0, 2, 1)
(Batch, 63, 15)          ← 63 channels, 15 timesteps
    │
    ▼ Conv Block 1: Conv1d(63→128, k=3) + BN + ReLU + MaxPool(2)
(Batch, 128, 7)          ← 15 → ⌊15/2⌋ = 7
    │
    ▼ Conv Block 2: Conv1d(128→256, k=3) + BN + ReLU + MaxPool(2)
(Batch, 256, 3)          ← 7 → ⌊7/2⌋ = 3
    │
    ▼ Conv Block 3: Conv1d(256→128, k=3) + BN + ReLU
(Batch, 128, 3)          ← giữ nguyên chiều dài
    │
    ▼ AdaptiveAvgPool1d(1)
(Batch, 128, 1)          ← Global Average Pooling
    │
    ▼ Flatten → Linear(128→64) → ReLU → Dropout(0.3) → Linear(64→num_classes)
(Batch, num_classes)
```

### Tại sao permute?
- `Conv1d` trong PyTorch kỳ vọng input có shape `(Batch, Channels, Length)`
- Dữ liệu gốc có shape `(Batch, 15_frames, 63_features)`
- Ta cần hoán vị thành `(Batch, 63_features, 15_frames)` để:
  - **63 features** đóng vai trò **channels** (như 3 kênh RGB trong ảnh)
  - **15 frames** đóng vai trò **chiều dài** mà kernel trượt qua
- Khi kernel trượt qua chiều thời gian, nó nhìn **tất cả 63 tọa độ đồng thời** tại mỗi cửa sổ 3-frame

In [ ]:
class GestureCNN1D(nn.Module):
    """1D-CNN cho phân loại chuỗi cử chỉ tay.
    
    Input:  (Batch, 15, 63)  – chuỗi 15 frame, mỗi frame 63 tọa độ
    Output: (Batch, num_classes)
    """

    def __init__(self, num_classes: int):
        super(GestureCNN1D, self).__init__()

        # ── Conv Block 1 ──
        self.conv_block1 = nn.Sequential(
            nn.Conv1d(in_channels=63, out_channels=128, kernel_size=3, padding=1),
            nn.BatchNorm1d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool1d(kernel_size=2)   # 15 → 7
        )

        # ── Conv Block 2 ──
        self.conv_block2 = nn.Sequential(
            nn.Conv1d(in_channels=128, out_channels=256, kernel_size=3, padding=1),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.MaxPool1d(kernel_size=2)   # 7 → 3
        )

        # ── Conv Block 3 ──
        self.conv_block3 = nn.Sequential(
            nn.Conv1d(in_channels=256, out_channels=128, kernel_size=3, padding=1),
            nn.BatchNorm1d(128),
            nn.ReLU(inplace=True)
        )

        # ── Global Average Pooling ──
        self.gap = nn.AdaptiveAvgPool1d(1)  # (Batch, 128, 3) → (Batch, 128, 1)

        # ── Classifier Head ──
        self.classifier = nn.Sequential(
            nn.Linear(128, 64),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        """
        Args:
            x: Tensor shape (Batch, 15, 63)
        Returns:
            logits: Tensor shape (Batch, num_classes)
        """
        # Hoán vị: (Batch, 15, 63) → (Batch, 63, 15)
        x = x.permute(0, 2, 1)

        # Trích xuất đặc trưng thời gian
        x = self.conv_block1(x)   # (Batch, 128, 7)
        x = self.conv_block2(x)   # (Batch, 256, 3)
        x = self.conv_block3(x)   # (Batch, 128, 3)

        # Global Average Pooling
        x = self.gap(x)           # (Batch, 128, 1)
        x = x.squeeze(-1)         # (Batch, 128)

        # Phân loại
        logits = self.classifier(x)  # (Batch, num_classes)
        return logits

In [ ]:
# ── Khởi tạo mô hình ──
model = GestureCNN1D(num_classes=num_classes).to(device)

# ── Đếm tham số ──
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print('🏗️  Kiến trúc mô hình:')
print('=' * 60)
print(model)
print('=' * 60)
print(f'📊 Tổng tham số:        {total_params:>10,}')
print(f'   Trainable:           {trainable_params:>10,}')
print(f'   Số lớp (classes):    {num_classes}')

In [ ]:
# ── Kiểm tra forward pass ──
with torch.no_grad():
    dummy_input = torch.randn(4, 15, 63).to(device)
    dummy_output = model(dummy_input)
    print(f'✅ Forward pass thành công!')
    print(f'   Input:  {dummy_input.shape}')    # (4, 15, 63)
    print(f'   Output: {dummy_output.shape}')   # (4, num_classes)

---
## 6. Cấu hình Huấn luyện

In [ ]:
# ── Hyperparameters ──
LEARNING_RATE = 1e-3
NUM_EPOCHS = 100

# ── Loss & Optimizer ──
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

print(f'⚙️  Cấu hình huấn luyện:')
print(f'   Optimizer:     Adam')
print(f'   Learning Rate: {LEARNING_RATE}')
print(f'   Loss:          CrossEntropyLoss')
print(f'   Epochs:        {NUM_EPOCHS}')
print(f'   Batch Size:    {BATCH_SIZE}')
print(f'   Device:        {device}')

---
## 7. Vòng lặp Huấn luyện

Chiến lược lưu mô hình: **Early Saving** – lưu trọng số mỗi khi Validation Accuracy đạt kỷ lục mới.

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    """Huấn luyện mô hình qua một epoch."""
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for X_batch, y_batch in loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        # Forward
        logits = model(X_batch)
        loss = criterion(logits, y_batch)

        # Backward
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Thống kê
        running_loss += loss.item() * X_batch.size(0)
        preds = logits.argmax(dim=1)
        correct += (preds == y_batch).sum().item()
        total += X_batch.size(0)

    epoch_loss = running_loss / total
    epoch_acc = correct / total
    return epoch_loss, epoch_acc


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    """Đánh giá mô hình trên tập validation/test."""
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    for X_batch, y_batch in loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        logits = model(X_batch)
        loss = criterion(logits, y_batch)

        running_loss += loss.item() * X_batch.size(0)
        preds = logits.argmax(dim=1)
        correct += (preds == y_batch).sum().item()
        total += X_batch.size(0)

    epoch_loss = running_loss / total
    epoch_acc = correct / total
    return epoch_loss, epoch_acc


print('✅ Hàm train_one_epoch() và evaluate() đã sẵn sàng.')

In [ ]:
# ══════════════════════════════════════════════════════
#                  VÒNG LẶP HUẤN LUYỆN
# ══════════════════════════════════════════════════════

history = {
    'train_loss': [], 'val_loss': [],
    'train_acc': [],  'val_acc': []
}

best_val_acc = 0.0
best_epoch = 0
start_time = time.time()

print(f'🚀 Bắt đầu huấn luyện {NUM_EPOCHS} epochs...')
print(f'{"Epoch":>6s} │ {"Train Loss":>10s} {"Train Acc":>10s} │ {"Val Loss":>10s} {"Val Acc":>10s} │ {"Status":>10s}')
print('─' * 75)

for epoch in range(1, NUM_EPOCHS + 1):
    # Huấn luyện
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)

    # Đánh giá
    val_loss, val_acc = evaluate(model, val_loader, criterion, device)

    # Ghi lịch sử
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)

    # Lưu best model
    status = ''
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_epoch = epoch
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_acc': val_acc,
            'val_loss': val_loss,
            'num_classes': num_classes,
            'label_classes': list(le.classes_),
            'total_params': total_params,
        }, BEST_MODEL_PATH)
        status = '✨ BEST'

    # In tiến trình (mỗi 10 epoch hoặc khi có best)
    if epoch % 10 == 0 or epoch == 1 or status:
        print(f'{epoch:>6d} │ {train_loss:>10.4f} {train_acc:>9.2%} │ {val_loss:>10.4f} {val_acc:>9.2%} │ {status:>10s}')

elapsed = time.time() - start_time
print('─' * 75)
print(f'⏱️  Thời gian huấn luyện: {elapsed:.1f}s ({elapsed/60:.1f} phút)')
print(f'🏆 Best Validation Accuracy: {best_val_acc:.2%} tại epoch {best_epoch}')
print(f'💾 Trọng số tốt nhất đã lưu: {BEST_MODEL_PATH}')

---
## 8. Trực quan hóa Quá trình Huấn luyện

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
epochs_range = range(1, NUM_EPOCHS + 1)

# ── Loss Curves ──
ax1 = axes[0]
ax1.plot(epochs_range, history['train_loss'], color=COLORS['train_loss'], label='Train Loss', alpha=0.9)
ax1.plot(epochs_range, history['val_loss'],   color=COLORS['val_loss'],   label='Val Loss',   alpha=0.9)
ax1.axvline(x=best_epoch, color='gray', linestyle='--', alpha=0.5, label=f'Best epoch ({best_epoch})')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('📉 Training & Validation Loss')
ax1.legend(loc='upper right')
ax1.set_xlim(1, NUM_EPOCHS)

# ── Accuracy Curves ──
ax2 = axes[1]
ax2.plot(epochs_range, [a * 100 for a in history['train_acc']], color=COLORS['train_acc'], label='Train Acc', alpha=0.9)
ax2.plot(epochs_range, [a * 100 for a in history['val_acc']],   color=COLORS['val_acc'],   label='Val Acc',   alpha=0.9)
ax2.axvline(x=best_epoch, color='gray', linestyle='--', alpha=0.5, label=f'Best epoch ({best_epoch})')
ax2.axhline(y=best_val_acc * 100, color=COLORS['val_acc'], linestyle=':', alpha=0.4)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy (%)')
ax2.set_title('📈 Training & Validation Accuracy')
ax2.legend(loc='lower right')
ax2.set_xlim(1, NUM_EPOCHS)

fig.suptitle('1D-CNN – Quá trình Huấn luyện', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## 9. Đánh giá trên Tập Test

Tải lại trọng số tốt nhất và đánh giá trên tập test (chưa từng được mô hình nhìn thấy).

In [ ]:
# ── Tải best model ──
checkpoint = torch.load(BEST_MODEL_PATH, map_location=device, weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])
print(f'✅ Đã tải trọng số từ epoch {checkpoint["epoch"]} (Val Acc: {checkpoint["val_acc"]:.2%})')

# ── Đánh giá trên Test set ──
test_loss, test_acc = evaluate(model, test_loader, criterion, device)

print(f'\n{"═" * 50}')
print(f'  🧪 KẾT QUẢ TRÊN TẬP TEST')
print(f'{"═" * 50}')
print(f'  Test Loss:     {test_loss:.4f}')
print(f'  Test Accuracy: {test_acc:.2%}')
print(f'{"═" * 50}')

In [ ]:
# ── Confusion Matrix & Classification Report ──
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

# Thu thập predictions
all_preds = []
all_labels = []

model.eval()
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch = X_batch.to(device)
        logits = model(X_batch)
        preds = logits.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(y_batch.numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

# ── Classification Report ──
print('📋 Classification Report:')
print('─' * 60)
print(classification_report(all_labels, all_preds, target_names=le.classes_))

# ── Confusion Matrix ──
cm = confusion_matrix(all_labels, all_preds)

fig, ax = plt.subplots(figsize=(max(8, num_classes), max(6, num_classes * 0.8)))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_,
            ax=ax, linewidths=0.5, linecolor='white')
ax.set_xlabel('Predicted Label', fontsize=12)
ax.set_ylabel('True Label', fontsize=12)
ax.set_title('🔍 Confusion Matrix – 1D-CNN (Test Set)', fontsize=14, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

---
## 10. So sánh MLP vs 1D-CNN

### Phân tích sự khác biệt kiến trúc:

| Tiêu chí | MLP (Tuần 2-3) | 1D-CNN (Tuần 4) |
|----------|----------------|------------------|
| **Input** | 1 frame → vector 63 chiều | 15 frames → ma trận (15, 63) |
| **Cách nhìn** | Tư thế tĩnh tại một thời điểm | Chuyển động qua chuỗi thời gian |
| **Trích xuất đặc trưng** | Fully-connected (global) | Tích chập cục bộ (local patterns) |
| **Parameter Sharing** | Không | Có (cùng kernel áp dụng cho mọi vị trí) |
| **Bất biến vị trí** | Không | Có (cùng pattern ở đầu/giữa/cuối chuỗi đều được phát hiện) |

Conv1d có lợi thế **chia sẻ tham số (weight sharing)**: cùng một bộ lọc được áp dụng tại mọi vị trí trên chuỗi thời gian, giúp mô hình học hiệu quả hơn với ít tham số hơn.

In [ ]:
# ══════════════════════════════════════════════════════
#        BẢNG SO SÁNH MLP vs 1D-CNN
# ══════════════════════════════════════════════════════

# ── Giá trị CNN (từ quá trình huấn luyện hiện tại) ──
cnn_test_acc = test_acc
cnn_params = total_params
cnn_best_epoch = best_epoch

# ── Thử tải giá trị MLP từ file checkpoint ──
mlp_model_path = os.path.join(MODEL_DIR, 'best_mlp.pth')
mlp_test_acc = None
mlp_params = None
mlp_best_epoch = None

if os.path.exists(mlp_model_path):
    try:
        mlp_ckpt = torch.load(mlp_model_path, map_location='cpu', weights_only=False)
        mlp_test_acc = mlp_ckpt.get('val_acc', None)
        mlp_params = mlp_ckpt.get('total_params', None)
        mlp_best_epoch = mlp_ckpt.get('epoch', None)
        print('✅ Đã tải thông tin MLP từ checkpoint.')
    except Exception as e:
        print(f'⚠️  Không thể tải MLP checkpoint: {e}')

# ── Sử dụng placeholder nếu không tải được ──
if mlp_test_acc is None:
    mlp_test_acc = '---'
    mlp_params = '---'
    mlp_best_epoch = '---'
    print('ℹ️  Không tìm thấy MLP checkpoint. Sử dụng placeholder.')
    print(f'   Hãy thay thế các giá trị "---" bằng kết quả thực tế từ Tuan2_3_Baseline_MLP.ipynb')

# ── In bảng so sánh ──
print(f'\n{"═" * 65}')
print(f'{"BẢNG SO SÁNH: MLP vs 1D-CNN":^65}')
print(f'{"═" * 65}')
print(f'{"Tiêu chí":<30s} │ {"MLP (Tuần 2-3)":>15s} │ {"1D-CNN (Tuần 4)":>15s}')
print(f'{"─" * 30}─┼─{"─" * 15}─┼─{"─" * 15}')

# Test Accuracy
mlp_acc_str = f'{mlp_test_acc:.2%}' if isinstance(mlp_test_acc, float) else str(mlp_test_acc)
cnn_acc_str = f'{cnn_test_acc:.2%}'
print(f'{"Test Accuracy":<30s} │ {mlp_acc_str:>15s} │ {cnn_acc_str:>15s}')

# Total Parameters
mlp_params_str = f'{mlp_params:>,d}' if isinstance(mlp_params, int) else str(mlp_params)
cnn_params_str = f'{cnn_params:>,d}'
print(f'{"Total Parameters":<30s} │ {mlp_params_str:>15s} │ {cnn_params_str:>15s}')

# Best Epoch
mlp_epoch_str = str(mlp_best_epoch)
cnn_epoch_str = str(cnn_best_epoch)
print(f'{"Best Val Acc @ Epoch":<30s} │ {mlp_epoch_str:>15s} │ {cnn_epoch_str:>15s}')

# Input Type
print(f'{"Input":<30s} │ {"1 frame (63,)":>15s} │ {"15 frames (15,63)":>15s}')

print(f'{"═" * 65}')

# ── Nhận xét tự động ──
if isinstance(mlp_test_acc, float):
    diff = cnn_test_acc - mlp_test_acc
    if diff > 0:
        print(f'\n🎯 1D-CNN vượt trội hơn MLP: +{diff:.2%} accuracy')
        print(f'   → Mô hình CNN tận dụng được thông tin thời gian từ 15 frame.')
    elif diff < 0:
        print(f'\n⚠️  MLP có accuracy cao hơn CNN: {abs(diff):.2%}')
        print(f'   → Có thể cần điều chỉnh hyperparameters hoặc thêm dữ liệu.')
    else:
        print(f'\n🔄 Hai mô hình có accuracy tương đương.')

---
## 📝 Tổng kết Tuần 4

### Những gì đã học:
1. **Conv1d cho dữ liệu chuỗi thời gian:** Bằng cách permute tensor từ `(B, 15, 63)` thành `(B, 63, 15)`, ta biến bài toán thành "trượt bộ lọc qua chiều thời gian" – mỗi bộ lọc nhìn vào cửa sổ 3-frame liên tiếp trên tất cả 63 tọa độ.

2. **Trích xuất đặc trưng phân cấp:** 
   - Block 1 phát hiện chuyển động cơ bản (3-frame patterns)
   - Block 2 kết hợp thành patterns phức tạp hơn (6-frame effective receptive field)
   - Block 3 tinh chỉnh đặc trưng trước khi pooling

3. **Global Average Pooling:** Thay vì flatten toàn bộ feature map (dễ overfit), GAP lấy trung bình mỗi kênh thành một số duy nhất – giảm tham số classifier đáng kể.

4. **So sánh với MLP:** CNN sử dụng toàn bộ 15 frame, có khả năng phát hiện **mẫu chuyển động cục bộ** mà MLP (chỉ nhìn 1 frame) không thể thấy được.

### Bước tiếp theo (Tuần 5-6):
- **Transfer Learning:** Thử nghiệm đóng băng/fine-tune trọng số
- **LSTM:** Mạng hồi quy để theo dõi sự thay đổi tọa độ qua toàn bộ chuỗi 15 frame